# LaLiga football match forecasting dataset

This notebook downloads the reproducible v0.2.0 match-level foundation, builds leakage-safe pre-match features, adds the available StatsBomb historical LaLiga lineups, events, players, and managers, evaluates a small baseline, and publishes the dataset. The publication cell reads `HF_TOKEN` from Colab Secrets; it never prompts for or prints the token.

In [ ]:
import os
os.chdir('/content')
!pip install -q -r https://raw.githubusercontent.com/EF-Code/laliga-match-forecasting/main/requirements-colab.txt

from pathlib import Path
import shutil

repo_dir = Path('/content/laliga-match-forecasting')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone -q https://github.com/EF-Code/laliga-match-forecasting.git /content/laliga-match-forecasting
os.chdir('/content/laliga-match-forecasting')

In [ ]:
# Read HF_TOKEN in the notebook kernel, then pass it in memory to the Hub uploader.
# The token is never prompted for, printed, or written to disk.
import sys
import importlib
from pathlib import Path
from google.colab import userdata

importlib.invalidate_caches()
sys.path.insert(0, '/content/laliga-match-forecasting/src')
import laliga_forecasting.build_dataset as builder

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add a write-capable HF_TOKEN secret in Colab and enable notebook access')

output_dir = Path('/content/laliga-output')
matches = builder.fetch_all_seasons()
pre_match, observations, team_stats = builder.build_pre_match_dataset(matches)
statsbomb_bundle = builder.build_statsbomb_bundle(output_dir / 'statsbomb-cache')
paths = builder.write_outputs(output_dir, pre_match, observations, team_stats, statsbomb_bundle)
repo_id = builder.publish_to_hub(output_dir, pre_match, paths, token=hf_token)
print(f'MATCH_ROWS {len(matches)}')
print(f'PRE_MATCH_ROWS {len(pre_match)}')
print(f'TEAM_MATCH_ROWS {len(team_stats)}')
for artifact_name, frame in statsbomb_bundle.items():
    print(f'{artifact_name.upper()}_ROWS {len(frame)}')
print(f'HF_DATASET_REPO https://huggingface.co/datasets/{repo_id}')
del hf_token

In [ ]:
!PYTHONPATH=src python -m laliga_forecasting.train_baseline --input /content/laliga-output/pre_match_forecasting.parquet --output /content/laliga-output/baseline_metrics.json